In [1]:
import time
import csv
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

In [2]:
OUTPUT_FILE = "IndoECommerce.csv"
MAX_PAGES = 100

In [3]:
URLS = [
    "https://www.tokopedia.com/glad2glowindo-443/new-glad2glow-glowing-bundle-4in1-niacinamide-milk-cleanser-body-serum-brightening-toner-moisturizer-serum-extract-hitam-mencerahkan-wajah-mencerahkan-melembabkan-kulit-skincare-skincare-memutihkan-dan-glowing-murah-bpom-1733567466432136482?extParam=ivf%3Dtrue%26keyword%3Dskincare%26search_id%3D202512190333193AD7C2EBB96FCB3B8M00%26src%3Dsearch&t_id=1766115205705&t_st=1&t_pp=search_result&t_efo=search_pure_goods_card&t_ef=goods_search&t_sm=&t_spt=search_result",
    "https://tk.tokopedia.com/ZSPtJLbB6/",
    "https://tk.tokopedia.com/ZSPtJCEyR/",
    "https://tk.tokopedia.com/ZSPtJshRR/",
    "https://tk.tokopedia.com/ZSPtJWnym/",
    "https://tk.tokopedia.com/ZSPtrGaUd/",
    "https://tk.tokopedia.com/ZSPtrfqpn/",
    "https://tk.tokopedia.com/ZSPthLD3f/",
    "https://tk.tokopedia.com/ZSPthrCDB/",
    "https://tk.tokopedia.com/ZSPthhom6/",
    "https://tk.tokopedia.com/ZSPth2VWJ/",
    "https://tk.tokopedia.com/ZSPtroj9n/",
    "https://tk.tokopedia.com/ZSPth8J2k/",
    "https://tk.tokopedia.com/ZSPthMAqR/",
    "https://tk.tokopedia.com/ZSPthjbSK/",
    "https://tk.tokopedia.com/ZSPthjbSK/",
    "https://tk.tokopedia.com/ZSPthrTtP/"   
]

# **First Scrapping**

In [4]:
def initialize_csv(filename):
    """Initialize CSV file with headers if it doesn't exist"""
    file_exists = os.path.isfile(filename)
    if not file_exists:
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Ulasan', 'Rating', 'Membantu'])
        print(f"Created new file: {filename}")
    else:
        print(f"Appending to existing file: {filename}")

def append_to_csv(filename, data):
    """Append data to CSV file"""
    with open(filename, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerows(data)

def setup_driver():
    """Setup and return Chrome WebDriver"""
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    return webdriver.Chrome(options=options)

def click_initial_button(driver):
    """Click the initial button if present"""
    try:
        button = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.css-11hzwo5 > button[type='button']"))
        )
        button.click()
        print("✓ Initial button clicked")
        time.sleep(2)
    except:
        print("⚠ Initial button not found, continuing...")

def navigate_to_reviews(driver):
    """Navigate to the reviews section"""
    driver.execute_script("window.scrollBy(0, 500);")
    time.sleep(2)
    
    try:
        review_button = driver.find_element(By.CSS_SELECTOR, "button[data-testid='review']")
        review_button.click()
        time.sleep(3)
        print("✓ Navigated to reviews section")
        return True
    except:
        print("✗ Could not find review button")
        return False

def extract_reviews(driver, url):
    """Extract reviews from current page"""
    soup = BeautifulSoup(driver.page_source, "html.parser")
    containers = soup.findAll('article', attrs={'class': 'css-15m2bcr'})
    
    page_data = []
    for container in containers:
        try:
            # Extract review text
            review = container.find('span', attrs={'data-testid': 'lblItemUlasan'}).text
            
            # Extract rating
            rating_div = container.find('div', attrs={'data-testid': 'icnStarRating'})
            aria_label = rating_div.get('aria-label')
            rating_value = int(aria_label.split()[1])
            
            # Extract helpful count
            helpful_element = container.find('span', class_='css-q2y3yl')
            if helpful_element:
                helpful_text = helpful_element.text.strip()
                helpful_count = 0 if helpful_text == "Membantu" else int(helpful_text.split()[0])
            else:
                helpful_count = 0
            
            page_data.append([review, rating_value, helpful_count])
        except (AttributeError, ValueError, IndexError):
            continue
    
    return page_data

def click_next_button(driver):
    """Click the next page button"""
    try:
        next_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button.css-dzvl4q-unf-pagination-item[aria-label='Laman berikutnya']"))
        )
        next_button.click()
        time.sleep(3)
        return True
    except:
        return False

def scrape_url(url, driver):
    """Scrape reviews from a single URL"""
    print(f"\n{'='*60}")
    print(f"Scraping: {url[:80]}...")
    print(f"{'='*60}")
    
    driver.get(url)
    time.sleep(3)
    
    # Handle initial popup/button
    click_initial_button(driver)
    
    # Navigate to reviews
    if not navigate_to_reviews(driver):
        return []
    
    all_data = []
    page_count = 0
    
    # Scrape reviews across pages
    for page in range(MAX_PAGES):
        print(f"Scraping page {page + 1}...", end=" ")
        
        # Extract reviews from current page
        page_data = extract_reviews(driver, url)
        all_data.extend(page_data)
        print(f"✓ Found {len(page_data)} reviews")
        
        # Save data incrementally every 5 pages
        if len(all_data) >= 50 and len(all_data) % 50 < 10:
            append_to_csv(OUTPUT_FILE, all_data)
            print(f"💾 Saved {len(all_data)} reviews to file")
            all_data = []
        
        # Try to go to next page
        if not click_next_button(driver):
            print("⚠ No more pages available")
            break
        
        page_count += 1
    
    # Save remaining data
    if all_data:
        append_to_csv(OUTPUT_FILE, all_data)
        print(f"💾 Saved final {len(all_data)} reviews to file")
    
    return page_count + 1

In [5]:
def main():
    """Main function to orchestrate the scraping"""
    print("🚀 Starting Tokopedia Review Scraper")
    print(f"📊 Output file: {OUTPUT_FILE}")
    print(f"🔗 URLs to scrape: {len(URLS)}")
    
    # Initialize CSV file
    initialize_csv(OUTPUT_FILE)
    
    # Setup driver
    driver = setup_driver()
    
    try:
        total_pages = 0
        for idx, url in enumerate(URLS, 1):
            print(f"\n[{idx}/{len(URLS)}] Processing URL...")
            pages = scrape_url(url, driver)
            total_pages += pages
            
            # Wait between URLs to avoid detection
            if idx < len(URLS):
                print("\n⏳ Waiting 10 seconds before next URL...")
                time.sleep(10)
        
        print(f"\n{'='*60}")
        print(f"✅ Scraping completed!")
        print(f"📄 Total pages scraped: {total_pages}")
        print(f"💾 Data saved to: {OUTPUT_FILE}")
        print(f"{'='*60}")
        
    except Exception as e:
        print(f"\n✗ Error occurred: {e}")
    finally:
        driver.quit()
        print("🔒 Browser closed")

if __name__ == "__main__":
    main()

🚀 Starting Tokopedia Review Scraper
📊 Output file: IndoECommerce.csv
🔗 URLs to scrape: 17
Appending to existing file: IndoECommerce.csv

[1/17] Processing URL...

Scraping: https://www.tokopedia.com/glad2glowindo-443/new-glad2glow-glowing-bundle-4in1-ni...
⚠ Initial button not found, continuing...
✓ Navigated to reviews section
Scraping page 1... ✓ Found 10 reviews
Scraping page 2... ✓ Found 10 reviews
Scraping page 3... ✓ Found 10 reviews
Scraping page 4... ✓ Found 10 reviews
Scraping page 5... ✓ Found 10 reviews
💾 Saved 50 reviews to file
Scraping page 6... ✓ Found 10 reviews
Scraping page 7... ✓ Found 10 reviews
Scraping page 8... ✓ Found 10 reviews
Scraping page 9... ✓ Found 10 reviews
Scraping page 10... ✓ Found 10 reviews
💾 Saved 50 reviews to file
Scraping page 11... ✓ Found 10 reviews
Scraping page 12... ✓ Found 10 reviews
Scraping page 13... ✓ Found 10 reviews
Scraping page 14... ✓ Found 10 reviews
Scraping page 15... ✓ Found 10 reviews
💾 Saved 50 reviews to file
Scraping page

# **Test Scrape Code**

In [6]:
df = pd.read_csv("IndoECommerce.csv")
print(df.shape)
df

(10559, 3)


,Ulasan,Rating,Membantu
0,Barang pesanan datang sesuai jadwal \nBarang p...,5,0
1,Bahan: Bagus\nManfaat produk: Sangat baik\nJen...,5,0
2,Bahan: bagus\nManfaat produk: mencerahkan\nJen...,5,1
3,Bahan: Sesuai pemesanan\nManfaat produk: Mence...,5,0
4,Bahan: Bagus sdah berlangganan\nManfaat produk...,5,0
...,...,...,...
10554,Sesuai harga,5,0
10555,nicee,5,0
10556,Nice,5,0
10557,"Sesuai harga, dan cocok untuk setup minimalis",5,0
